# EV scheduling: two models, quantum encodings and independent checks

This notebook consolidates the station-booking and final-charge-fairness models into one executable experiment. All inputs are mock teaching data. **Every quantum execution is a classical simulation; no IBM account, API key, cloud job or real charger is used.**

Use **Restart Kernel → Run All**. The full model sources are included below; there are no repository-relative source downloads. Temporary support modules are created in an isolated temporary directory and are removed at the end.

| Model | Decision and objective | Encoding |
|---|---|---|
| Booking | One station/hour per car; minimize electricity bill + weighted waiting | 5-qubit QUBO/Ising |
| Fair charging | Four 15-minute joint power allocations; maximize lower final SoC, then kWh | 12-qubit diagonal rank cost, **not a quadratic QUBO** |

The earlier eight-qubit shared-power/completion-time model has a different objective. It is not substituted for fairness here.

Mathematical correctness, sampled optimizer quality and physical usefulness are separate questions. Classical enumeration is the reference, never a hidden replacement for a failed quantum sample.

## 1. Reproducible environment

The validation environment uses Qiskit 2.6.1, NumPy 2.5.4, SciPy 1.18.1, Aer 0.17.2 and Runtime 0.50.1. Use the repository's locked `uv` environment when available. This notebook also works independently in Jupyter or an online Python notebook host, provided these packages and `uv` are installed.

If packages are absent, the next cell installs the pinned versions into the active kernel using `uv pip`. If `uv` is absent, install it using the [official instructions](https://docs.astral.sh/uv/getting-started/installation/) and rerun. On an online host, each viewer needs their own runtime; sharing the notebook does not share your running VM. See the [Colab FAQ](https://research.google.com/colaboratory/faq.html).

IBM's cloud simulators were [retired in 2024](https://quantum.cloud.ibm.com/docs/en/guides/local-simulators). This notebook is portable Python simulation, not an IBM-hosted mock project.

In [1]:
import sys, shutil, subprocess, importlib.metadata as metadata
REQUIRED = {"qiskit":"2.6.1", "numpy":"2.5.4", "scipy":"1.18.1",
            "qiskit-aer":"0.17.2", "qiskit-ibm-runtime":"0.50.1"}
def installed(name):
    try: return metadata.version(name)
    except metadata.PackageNotFoundError: return None
mismatched = [f"{name}=={version}" for name,version in REQUIRED.items() if installed(name)!=version]
if mismatched and any(name in sys.modules for name in ('qiskit','numpy','scipy')):
    raise RuntimeError('Restart the kernel before installing different numerical package versions, then Run All.')
if mismatched:
    uv = shutil.which("uv")
    if uv is None:
        raise RuntimeError("Install uv from https://docs.astral.sh/uv/getting-started/installation/ and rerun this cell.")
    subprocess.run([uv,"pip","install","--python",sys.executable,*mismatched],check=True)
print({name:installed(name) for name in REQUIRED})

{'qiskit': '2.6.1', 'numpy': '2.5.4', 'scipy': '1.18.1', 'qiskit-aer': '0.17.2', 'qiskit-ibm-runtime': '0.50.1'}


In [2]:
from pathlib import Path
import tempfile, os, json
from IPython.display import display, Markdown
WORKSPACE = tempfile.TemporaryDirectory(prefix="ev-scheduling-notebook-")
PROJECT = Path(WORKSPACE.name)
SIM_DIR = PROJECT / "simulations"
SIM_DIR.mkdir()
CLASSICAL_PATH = str(PROJECT / "model.py")
BOOKING_PATH = str(SIM_DIR / "booking_model.py")
FAIRNESS_PATH = str(SIM_DIR / "fairness_model.py")
VALIDATION_PATH = str(SIM_DIR / "validate.py")
ENGINE_PATH = str(SIM_DIR / "engine.mjs")
COMPARE_PATH = str(SIM_DIR / "compare.mjs")
sys.path.insert(0,str(SIM_DIR))
print("Isolated temporary source workspace prepared.")

Isolated temporary source workspace prepared.


## 2. Classical fairness reference

For car $i$, $E_i=0.25\sum_t p_{it}$ kWh and $s_{i,final}=s_{i,start}+100E_i/C_i$. The objective is the ordered pair $(\min_{i\ present}s_{i,final},\sum_i E_i)$. Compare the first component first; energy breaks only exact ties.

The reference independently enumerates each car's whole-hour sequences, then checks station compatibility. The quantum encoding below instead decodes joint slot codes. This gives two different paths for checking the answer. All absent/full cars, power ceilings, station overload and battery headroom are enforced.

In [3]:
%%writefile $CLASSICAL_PATH
"""Exact classical reference. All arithmetic used for ranking is rational.

Run with uv run model.py, or pipe a JSON scenario to uv run model.py --stdin.
No third-party dependencies and no quantum execution.
"""
from fractions import Fraction as F
from itertools import product
import argparse
import json
import math
import sys

SLOTS = 4
DT = F(1, 4)
DEFAULT = {"cars": [
    {"capacity": 60, "soc": 60, "fast": True, "present": True},
    {"capacity": 80, "soc": 30, "fast": True, "present": True},
], "rate": 8}


def validate(scenario):
    if not isinstance(scenario, dict) or not isinstance(scenario.get("cars"), list) or len(scenario["cars"]) != 2:
        raise ValueError("Exactly two cars are required")
    for car in scenario["cars"]:
        if not isinstance(car, dict):
            raise ValueError("Each car must be an object")
        for key in ("capacity", "soc"):
            value = car.get(key)
            if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
                raise ValueError(f"{key} must be a finite number")
        if car["capacity"] <= 0 or not 0 <= car["soc"] <= 100:
            raise ValueError("Capacity must be positive and percentage between 0 and 100")
        if type(car.get("fast")) is not bool or type(car.get("present")) is not bool:
            raise ValueError("fast and present must be booleans")
    rate = scenario.get("rate", 8)
    if isinstance(rate, bool) or not isinstance(rate, (int, float)) or not math.isfinite(rate) or rate < 0:
        raise ValueError("Rate must be a finite nonnegative number")


def solve(scenario):
    validate(scenario)
    cars = scenario["cars"]
    sequences = []
    # Generate each car's whole-hour plans independently. Station conflicts are
    # checked only when combining the two lists (independent of the JS DFS).
    for car in cars:
        levels = [0] if not car["present"] or car["soc"] == 100 else ([0, 25, 50] if car["fast"] else [0, 25])
        headroom = F(str(car["capacity"])) * (1 - F(str(car["soc"])) / 100)
        sequences.append([p for p in product(levels, repeat=SLOTS) if sum(p) * DT <= headroom])
    best = None
    ties = feasible = 0
    winner = None
    for a, b in product(*sequences):
        if any(x + y > 50 for x, y in zip(a, b)):
            continue
        feasible += 1
        energy = [sum(a) * DT, sum(b) * DT]
        final = [F(str(c["soc"])) + 100 * e / F(str(c["capacity"])) for c, e in zip(cars, energy)]
        active = [s for c, s in zip(cars, final) if c["present"]]
        score = (min(active) if active else F(0), sum(energy))
        if best is None or score > best:
            best, winner, ties = score, (a, b, energy, final), 1
        elif score == best:
            ties += 1
    a, b, energy, final = winner
    return {"schedule": [list(p) for p in zip(a, b)], "energy": list(map(float, energy)),
            "final": list(map(float, final)), "minimum": float(best[0]) if any(c["present"] for c in cars) else None,
            "total": float(best[1]), "cost": float(best[1] * F(str(scenario.get("rate", 8)))),
            "feasible": feasible, "ties": ties}


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--stdin", action="store_true")
    args = parser.parse_args()
    try:
        print(json.dumps(solve(json.load(sys.stdin) if args.stdin else DEFAULT), indent=2))
    except (ValueError, TypeError, KeyError) as error:
        print(str(error), file=sys.stderr)
        sys.exit(1)


Writing temporary-source/model.py


## 3. Booking: physical options → QUBO → Ising

The fixed scenario has two 11 kW stations and two full-hour reservations. Travel, reserve, arrival and actual completion deadlines remove invalid individual options, leaving five. One bit selects each booking.

$F(x)=\sum_i c_i x_i + P[(x_0+x_1+x_2+x_3-1)^2+(x_4-1)^2+x_0x_4]$.

Costs $c_i$ are bill + weighted waiting, measured in preference points. Let $B$ be the sum of each car's maximum nonnegative option cost and choose $P=B+1$. Valid costs are at most $B$; any invalid state incurs at least $P$. Substitute $x_i=(1-Z_i)/2$ for the Ising operator. The complete implementation checks both expansions exactly.

In [4]:
%%writefile $BOOKING_PATH
"""Five-qubit reference bookings; all model construction is local and exact.

The optional scenario contains only waiting_weight in [0, 1]. Fixed mock
physical inputs deliberately match the teaching example, not a live station.
"""
from fractions import Fraction as F
from itertools import combinations, product
import argparse
import json
import math

VEHICLES = (
    dict(id="A", capacity=50, initial_percent=30, target_percent=50,
         available=465, deadline=600),
    dict(id="B", capacity=50, initial_percent=12, target_percent=30,
         available=470, deadline=540),
)
STATIONS = (dict(id="North", power=11, prices=(2, 1)),
            dict(id="South", power=11, prices=(3, 2)))
TRAVEL = {("A", "North"): (2, 10), ("A", "South"): (5, 15),
          ("B", "North"): (1, 5), ("B", "South"): (20, 30)}
SLOTS = (("early", 480, 540), ("late", 540, 600))


def _weight(scenario):
    if scenario is None:
        return F(1, 4)
    if not isinstance(scenario, dict) or set(scenario) - {"waiting_weight"}:
        raise ValueError("Only the reference scenario with waiting_weight is supported")
    value = scenario.get("waiting_weight", 0.25)
    if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value) or not 0 <= value <= 1:
        raise ValueError("waiting_weight must be a finite number from 0 to 1")
    return F(str(value))


def _candidate(vehicle, station, slot_index):
    slot_id, start, end = SLOTS[slot_index]
    distance, duration = TRAVEL[vehicle["id"], station["id"]]
    arrival = vehicle["available"] + duration
    arrival_energy = F(vehicle["capacity"] * vehicle["initial_percent"], 100) - F(distance, 5)
    required = F(vehicle["capacity"] * vehicle["target_percent"], 100) - arrival_energy
    completion = F(start) + 60 * required / station["power"]
    reasons = []
    if arrival_energy < F(vehicle["capacity"], 20):
        reasons.append("arrival reserve below 5 percent")
    if arrival > start:
        reasons.append("arrival after booking start")
    if required <= 0:
        reasons.append("no positive charge required")
    if required > F(station["power"] * (end - start), 60):
        reasons.append("required energy exceeds booking capacity")
    if completion > vehicle["deadline"]:
        reasons.append("completion after departure deadline")
    return dict(vehicle_id=vehicle["id"], station_id=station["id"],
                slot_id=slot_id, start_minute=start, end_minute=end,
                arrival_minute=arrival, arrival_energy_kwh=arrival_energy,
                charging_energy_kwh=required, completion_minute=completion,
                electricity_cost_rupees=required * station["prices"][slot_index],
                waiting_minutes=start-arrival, reasons=reasons)


def _options():
    candidates = [_candidate(v, s, t) for v in VEHICLES for s in STATIONS for t in range(2)]
    return [c for c in candidates if not c["reasons"]], [c for c in candidates if c["reasons"]]


def _parameters(scenario=None):
    weight = _weight(scenario)
    options, _ = _options()
    groups = [[i for i, o in enumerate(options) if o["vehicle_id"] == v["id"]] for v in VEHICLES]
    scores = [o["electricity_cost_rupees"] + weight * o["waiting_minutes"] for o in options]
    bound = sum(max(scores[i] for i in group) for group in groups)
    penalty = bound + 1
    conflicts = [(i, j) for i, j in combinations(range(len(options)), 2)
                 if options[i]["vehicle_id"] != options[j]["vehicle_id"]
                 and (options[i]["station_id"], options[i]["slot_id"]) ==
                     (options[j]["station_id"], options[j]["slot_id"])]
    pairs = {pair: 2 * penalty for group in groups for pair in combinations(group, 2)}
    for pair in conflicts:
        pairs[pair] = pairs.get(pair, F(0)) + penalty
    linear = [cost - penalty for cost in scores]
    constant = penalty * len(groups)
    z = [-value / 2 for value in linear]
    zz = {pair: value / 4 for pair, value in pairs.items()}
    offset = constant + sum(linear) / 2 + sum(pairs.values()) / 4
    for (i, j), value in zz.items():
        z[i] -= value
        z[j] -= value
    return options, groups, scores, bound, penalty, conflicts, linear, pairs, constant, z, zz, offset


def _bits(index):
    if isinstance(index, bool) or not isinstance(index, int) or not 0 <= index < 32:
        raise ValueError("state index must be an integer in [0, 31]")
    return [(index >> i) & 1 for i in range(5)]


def _json_numbers(value):
    if isinstance(value, F):
        return float(value)
    if isinstance(value, dict):
        return {k: _json_numbers(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_numbers(v) for v in value]
    return value


def decode(index, scenario=None):
    bits = _bits(index)
    options, groups, scores, _, penalty, conflicts, *_ = _parameters(scenario)
    selected = [dict(o) for i, o in enumerate(options) if bits[i]]
    reasons = [f"Car {VEHICLES[n]['id']} needs exactly one booking"
               for n, g in enumerate(groups) if sum(bits[i] for i in g) != 1]
    if any(bits[i] and bits[j] for i, j in conflicts):
        reasons.append("North/early port is double booked")
    raw_score = sum(c * b for c, b in zip(scores, bits))
    units = sum((sum(bits[i] for i in g) - 1)**2 for g in groups) + sum(bits[i] * bits[j] for i, j in conflicts)
    return _json_numbers(dict(valid=not reasons, feasible=not reasons, reasons=reasons,
        bits=bits, booking_bitstring="".join(map(str, bits)), qiskit_bitstring=format(index, "05b"),
        schedule=selected, score=raw_score, penalty=penalty * units,
        energy=raw_score + penalty * units,
        total_electricity_cost_rupees=sum(o["electricity_cost_rupees"] for o in selected),
        total_waiting_minutes=sum(o["waiting_minutes"] for o in selected)))


def build_model(scenario=None):
    options, groups, scores, bound, penalty, conflicts, linear, pairs, constant, z, zz, offset = _parameters(scenario)
    energies = []
    feasible = []
    for index in range(32):
        bits = _bits(index)
        energies.append(float(constant + sum(c * b for c, b in zip(linear, bits)) +
                        sum(c * bits[i] * bits[j] for (i, j), c in pairs.items())))
        feasible.append(all(sum(bits[i] for i in g) == 1 for g in groups) and
                        not any(bits[i] and bits[j] for i, j in conflicts))
    optimum = min(e for e, allowed in zip(energies, feasible) if allowed)
    return dict(name="booking", num_qubits=5, energies=energies, feasible=feasible,
        metadata=_json_numbers(dict(waiting_weight=_weight(scenario), options=options,
            objective="electricity bill plus weighted waiting (preference points)",
            score_bound=bound, penalty=penalty, constant=constant, linear=linear,
            quadratic=[[i, j, c] for (i, j), c in pairs.items()],
            z=z, zz=[[i, j, c] for (i, j), c in zz.items()], offset=offset,
            scale=max([abs(c) for c in z] + [abs(c) for c in zz.values()]),
            exact_optimum=optimum,
            optimal_indices=[i for i, e in enumerate(energies) if feasible[i] and abs(e-optimum)<1e-9],
            encoding="one qubit per prevalidated full-hour booking", execution="local simulation only")))


def verify():
    """Compare physical choices, direct penalties, expanded QUBO and Ising exactly."""
    options, rejected = _options()
    assert len(options) == 5 and len(rejected) == 3
    assert [(o["vehicle_id"], o["station_id"], o["slot_id"]) for o in options] == [
        ("A", "North", "early"), ("A", "North", "late"),
        ("A", "South", "early"), ("A", "South", "late"), ("B", "North", "early")]
    # Independent physical pair enumerator uses candidate records, no penalties.
    checked = 0
    for step in range(101):
        scenario = {"waiting_weight": step/100}
        model = build_model(scenario)
        opts, groups, costs, bound, penalty, conflicts, linear, pairs, constant, z, zz, offset = _parameters(scenario)
        weight = _weight(scenario)
        legal = []
        for a, b in product([o for o in opts if o["vehicle_id"] == "A"], [o for o in opts if o["vehicle_id"] == "B"]):
            if (a["station_id"], a["slot_id"]) != (b["station_id"], b["slot_id"]):
                cost = a["electricity_cost_rupees"]+b["electricity_cost_rupees"] + weight*(a["waiting_minutes"]+b["waiting_minutes"])
                legal.append((a, b, cost))
        assert len(legal) == 3
        reference = min(c for _, _, c in legal)
        all_exact = []
        for index in range(32):
            bits = _bits(index)
            selected = [o for i, o in enumerate(opts) if bits[i]]
            physical_valid = len(selected) == 2 and {o["vehicle_id"] for o in selected} == {"A", "B"} and len({(o["station_id"], o["slot_id"]) for o in selected}) == 2
            direct = sum(c * b for c, b in zip(costs, bits)) + penalty * ((sum(bits[:4])-1)**2 + (bits[4]-1)**2 + bits[0]*bits[4])
            expanded = constant + sum(c*b for c, b in zip(linear, bits)) + sum(c*bits[i]*bits[j] for (i,j),c in pairs.items())
            spins = [1-2*b for b in bits]
            ising = offset + sum(c*s for c,s in zip(z,spins)) + sum(c*spins[i]*spins[j] for (i,j),c in zz.items())
            assert direct == expanded == ising
            assert model["energies"][index] == float(direct)
            assert model["feasible"][index] == physical_valid == decode(index,scenario)["valid"]
            if physical_valid:
                assert direct <= bound
            else:
                assert direct >= penalty > reference
            all_exact.append(direct)
            checked += 1
        assert min(all_exact) == reference
        assert model["metadata"]["exact_optimum"] == float(reference)
    default = decode(18)  # displayed x0..x4 = 01001; Qiskit counts = 10010
    assert default["booking_bitstring"] == "01001" and default["qiskit_bitstring"] == "10010"
    assert default["score"] == 46.3 and default["total_electricity_cost_rupees"] == 28.8
    assert decode(20,{"waiting_weight":1})["score"] == 56.4
    assert not decode(17)["valid"]  # conflicting x0 and x4
    assert decode(17)["energy"] == 99.35
    for bad in ([], {"cars": []}, {"waiting_weight": True}, {"waiting_weight":float("nan")},
                {"waiting_weight":float("inf")}, {"waiting_weight":-1}, {"waiting_weight":1.01},
                {"waiting_weight":"0.25"}):
        try:
            build_model(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"accepted invalid scenario {bad!r}")
    for bad in (-1,32,True,1.5,"10010"):
        try:
            decode(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(f"accepted invalid state {bad!r}")
    return {"encoded_state_weight_checks":checked,"default_optimum":default["score"],"feasible_states":3}


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--verify", action="store_true")
    args = parser.parse_args()
    if args.verify:
        print(json.dumps(verify(), sort_keys=True))
        print("MODEL_VERIFIED")
    else:
        print(json.dumps(build_model(), indent=2))


Writing temporary-source/booking_model.py


## 4. Fairness: a faithful bounded diagonal encoding

Three bits encode each interval. Codes 0–5 mean `(0,0), (25,0), (50,0), (0,25), (25,25), (0,50)` kW. Codes 6–7 are invalid. Four intervals use 12 qubits. Capabilities, presence and battery headroom impose further feasibility checks.

Rank distinct feasible lower final percentages from 0 to $m-1$. For rank $r$ and delivered 6.25 kWh block count $u\in[0,8]$, use $E=9(m-1-r)+8-u$. Invalid states have $E=9m$. A rank improvement dominates any energy tie-break; every invalid state costs more than every valid state.

**Limitation:** this constructs a 4,096-entry table by classical enumeration. It preserves every ordered objective comparison, but changes objective gaps and can change QAOA's search landscape. It is a transparent teaching Hamiltonian, not a scalable QUBO or a claim of quantum advantage.

In [5]:
%%writefile $FAIRNESS_PATH
"""Bounded 12-qubit diagonal encoding of final-charge fairness.

This is a small, exact lookup-table Hamiltonian for local simulation, not a
quadratic QUBO or a claim of efficient hardware synthesis. All model decisions
and objective ranks use exact rational arithmetic. No quantum service is used.
"""
from copy import deepcopy
from fractions import Fraction as F
from pathlib import Path
import argparse
import importlib.util
import json
import math
import random
import sys

DEFAULT = {"cars": [
    {"capacity": 60, "soc": 60, "fast": True, "present": True},
    {"capacity": 80, "soc": 30, "fast": True, "present": True},
], "rate": 8}
# Three little-endian bits encode one joint power allocation for one interval.
PAIRS = ((0, 0), (25, 0), (50, 0), (0, 25), (25, 25), (0, 50))
NUM_QUBITS = 12
STATE_COUNT = 1 << NUM_QUBITS


def _scenario(scenario):
    s = deepcopy(DEFAULT if scenario is None else scenario)
    if not isinstance(s, dict) or set(s) - {"cars", "rate"}:
        raise ValueError("Only cars and optional rate are supported")
    cars = s.get("cars")
    if not isinstance(cars, list) or len(cars) != 2:
        raise ValueError("Exactly two cars are required")
    for car in cars:
        if not isinstance(car, dict) or set(car) != {"capacity", "soc", "fast", "present"}:
            raise ValueError("Each car needs capacity, soc, fast and present")
        for key in ("capacity", "soc"):
            v = car[key]
            if type(v) not in (int, float) or not math.isfinite(v):
                raise ValueError(f"{key} must be a finite number")
        if car["capacity"] <= 0 or not 0 <= car["soc"] <= 100:
            raise ValueError("Capacity must be positive and SoC between 0 and 100")
        if type(car["fast"]) is not bool or type(car["present"]) is not bool:
            raise ValueError("Capability and presence must be booleans")
    rate = s.setdefault("rate", 8)
    if type(rate) not in (int, float) or not math.isfinite(rate) or rate < 0:
        raise ValueError("Rate must be finite and nonnegative")
    return s


def _evaluate(index, s):
    if type(index) is not int or not 0 <= index < STATE_COUNT:
        raise ValueError("Index must be an integer in [0, 4095]")
    codes = [(index >> (3 * t)) & 7 for t in range(4)]
    result = {"index": index, "bits": format(index, "012b"),
              "slot_codes": codes, "feasible": False}
    if any(code >= len(PAIRS) for code in codes):
        return {**result, "reason": "Unused slot code (6 or 7)"}
    schedule = [list(PAIRS[code]) for code in codes]
    result["schedule"] = schedule
    units = [sum(pair[i] for pair in schedule) // 25 for i in range(2)]
    for i, car in enumerate(s["cars"]):
        if not car["present"] and units[i]:
            return {**result, "reason": "Absent car receives power"}
        if not car["fast"] and any(pair[i] > 25 for pair in schedule):
            return {**result, "reason": "Vehicle power ceiling exceeded"}
    energy = [F(25, 4) * u for u in units]
    final = [F(str(c["soc"])) + 100 * e / F(str(c["capacity"]))
             for c, e in zip(s["cars"], energy)]
    if any(v > 100 for v in final):
        return {**result, "reason": "Battery would exceed 100 percent"}
    active = [v for c, v in zip(s["cars"], final) if c["present"]]
    minimum = min(active) if active else F(0)
    result.update(feasible=True, reason=None, energy=list(map(float, energy)),
                  final=list(map(float, final)),
                  minimum=float(minimum) if active else None,
                  total=float(sum(energy)), cost=float(sum(energy) * F(str(s["rate"]))),
                  total_units=sum(units),
                  objective_exact=[str(minimum), str(sum(energy))])
    return result


def decode(index, scenario=None):
    """Decode and physically validate an integer measurement outcome.

    bits uses Qiskit's display order q11...q0. The rightmost three bits describe
    interval zero; slot_codes and schedule are in chronological order.
    """
    return _evaluate(index, _scenario(scenario))


def build_model(scenario=None):
    """Return exact integer diagonal energies with the same lexicographic order.

    Rank the distinct achievable minimum final percentages, from low to high.
    At most eight 6.25-kWh blocks fit into the station's hour. A rank step of nine
    dominates every possible difference in block count; within one rank the
    lower energy corresponds exactly to more delivered energy. Every invalid
    state has energy 9*m, strictly above every valid energy (at most 9*m-1).
    """
    s = _scenario(scenario)
    results = [_evaluate(i, s) for i in range(STATE_COUNT)]
    minima = sorted({F(r["objective_exact"][0]) for r in results if r["feasible"]})
    ranks = {v: rank for rank, v in enumerate(minima)}
    invalid_energy = 9 * len(minima)
    energies = [9 * (len(minima) - 1 - ranks[F(r["objective_exact"][0])])
                + 8 - r["total_units"] if r["feasible"] else invalid_energy
                for r in results]
    minimum_energy = min(energies)
    optimal_indices = [i for i, e in enumerate(energies) if e == minimum_energy]
    return {"name": "fairness", "num_qubits": NUM_QUBITS,
            "energies": energies, "feasible": [r["feasible"] for r in results],
            "metadata": {"encoding": "diagonal-rank-cost-not-QUBO",
                         "scenario": s, "slot_pairs": list(map(list, PAIRS)),
                         "unused_slot_codes": [6, 7], "bits_per_slot": 3,
                         "minimum_ranks": [str(v) for v in minima],
                         "rank_weight": 9, "invalid_energy": invalid_energy,
                         "minimum_energy": minimum_energy,
                         "feasible_count": sum(r["feasible"] for r in results),
                         "optimal_indices": optimal_indices,
                         "best": results[optimal_indices[0]],
                         "bit_order": "integer bit i = qubit i; slot t = bits 3t..3t+2",
                         "execution": "local classical simulation of quantum circuit",
                         "limitation": "Bounded exhaustive diagonal construction; no scalable QUBO or hardware efficiency claim"}}


def _oracle(index, s):
    # Intentionally independent per-state physical calculation, without _evaluate.
    raw = format(index, "012b")
    codes = [int(raw[9 - 3 * t:12 - 3 * t], 2) for t in range(4)]
    if any(c not in range(6) for c in codes):
        return None
    plan = [((0, 0), (25, 0), (50, 0), (0, 25), (25, 25), (0, 50))[c] for c in codes]
    energies = []
    final = []
    for i, car in enumerate(s["cars"]):
        allowed = {0} if not car["present"] else {0, 25, 50} if car["fast"] else {0, 25}
        if any(pair[i] not in allowed or sum(pair) > 50 for pair in plan):
            return None
        e = sum(F(pair[i], 4) for pair in plan)
        end = F(str(car["soc"])) + 100 * e / F(str(car["capacity"]))
        if end > 100:
            return None
        energies.append(e)
        final.append(end)
    present = [v for c, v in zip(s["cars"], final) if c["present"]]
    return (min(present) if present else F(0), sum(energies))


def verify():
    root = Path(__file__).resolve().parents[1]
    spec = importlib.util.spec_from_file_location("independent_classical_reference", root / "model.py")
    reference = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(reference)
    rng = random.Random(20261101)
    cases = [deepcopy(DEFAULT)]
    for _ in range(95):
        cases.append({"cars": [{"capacity": rng.choice([40, 50, 60, 80]),
             "soc": rng.choice([0, 10, 30, 42.5, 60, 75, 90, 99, 100]),
             "fast": rng.choice([True, False]), "present": rng.choice([True, True, False])}
             for _ in range(2)], "rate": rng.choice([0, 8, 12.5])})
    for flags in ((False, False), (True, False), (True, True)):
        s = deepcopy(DEFAULT)
        for c, present in zip(s["cars"], flags):
            c.update(present=present, soc=99 if present else 0)
        cases.append(s)
    s = deepcopy(DEFAULT)
    s["cars"][0].update(capacity=50, soc=75)
    s["cars"][1]["present"] = False
    cases.append(s)
    for case in cases:
        m = build_model(case)
        oracle = [_oracle(i, case) for i in range(STATE_COUNT)]
        assert m["feasible"] == [v is not None for v in oracle]
        valid_indices = [i for i, v in enumerate(oracle) if v is not None]
        exact_best = max(oracle[i] for i in valid_indices)
        exact_winners = [i for i in valid_indices if oracle[i] == exact_best]
        assert m["metadata"]["optimal_indices"] == exact_winners
        # Compare every adjacent distinct objective, not only the optimum.
        objectives = {}
        for i in valid_indices:
            score = oracle[i]
            if score in objectives:
                assert m["energies"][i] == objectives[score]
            objectives[score] = m["energies"][i]
        ordered = sorted(objectives)
        assert all(objectives[a] > objectives[b] for a, b in zip(ordered, ordered[1:]))
        assert max(m["energies"][i] for i in valid_indices) < m["metadata"]["invalid_energy"]
        expected = reference.solve(case)
        assert m["metadata"]["feasible_count"] == expected["feasible"]
        assert len(exact_winners) == expected["ties"]
        winner = decode(exact_winners[0], case)
        for field in ("minimum", "total", "cost"):
            assert winner[field] == expected[field], (field, winner, expected)
    # Price is a report, not an objective term.
    expensive = deepcopy(DEFAULT)
    expensive["rate"] = 1000
    assert build_model(expensive)["energies"] == build_model(DEFAULT)["energies"]
    for index in (-1, STATE_COUNT, True, 1.5):
        try:
            decode(index)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid index was accepted")
    for field, value in (("capacity", 0), ("capacity", True), ("soc", 101),
                         ("soc", float("nan")), ("fast", 1), ("present", "yes")):
        bad = deepcopy(DEFAULT)
        bad["cars"][0][field] = value
        try:
            build_model(bad)
        except ValueError:
            pass
        else:
            raise AssertionError("Invalid scenario was accepted")
    # Positive controls ensure the physical validator detects encoded violations.
    assert not decode(6)["feasible"]
    absent = deepcopy(DEFAULT)
    absent["cars"][0]["present"] = False
    assert not decode(1, absent)["feasible"]
    slow = deepcopy(DEFAULT)
    slow["cars"][0]["fast"] = False
    assert not decode(2, slow)["feasible"]
    full = deepcopy(DEFAULT)
    full["cars"][0]["soc"] = 100
    assert not decode(1, full)["feasible"]
    default = build_model()
    assert default["metadata"]["best"]["minimum"] == 76.875
    return {"scenarios": len(cases), "states_per_scenario": STATE_COUNT,
            "basis_state_checks": len(cases) * STATE_COUNT,
            "default_feasible": default["metadata"]["feasible_count"],
            "default_optimal_states": len(default["metadata"]["optimal_indices"]),
            "default_best": default["metadata"]["best"]}


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--verify", action="store_true")
    parser.add_argument("--stdin", action="store_true", help="Read a scenario as JSON")
    args = parser.parse_args()
    if args.verify:
        print(json.dumps(verify(), indent=2))
        print("MODEL_VERIFIED")
    else:
        print(json.dumps(build_model(json.load(sys.stdin) if args.stdin else None)))


Writing temporary-source/fairness_model.py


## 5. Execute the mathematical audits

These checks compare physical feasibility, objective ordering, penalties and exact classical winners. They do not depend on sampling success. Positive controls intentionally test invalid booking conflicts, unused codes, absent charging, unsupported fast charging and overfill.

In [6]:
import booking_model, fairness_model
booking_checks=booking_model.verify()
fairness_checks=fairness_model.verify()
display(Markdown(f"**Booking:** {booking_checks['encoded_state_weight_checks']:,} exact state/weight comparisons passed."))
display(Markdown(f"**Fairness:** {fairness_checks['basis_state_checks']:,} state/scenario checks across {fairness_checks['scenarios']} scenarios passed."))
assert fairness_checks['default_feasible']==933
assert fairness_checks['default_optimal_states']==10
print("MATHEMATICAL_MODELS_VERIFIED")

**Booking:** 3,232 exact state/weight comparisons passed.

**Fairness:** 409,600 state/scenario checks across 100 scenarios passed.

MATHEMATICAL_MODELS_VERIFIED


## 6. Circuit construction and local training

From a uniform superposition, each layer applies $\exp(-i\gamma H_C/scale)$ and then $RX(2\beta)$ on every qubit. Booking uses RZ/RZZ cost terms; its constant is an irrelevant global phase. Fairness uses Qiskit's structured diagonal synthesis; no dense 4,096×4,096 matrix is built.

The vectorized training routine is checked against independently constructed Qiskit circuits. Three seeded COBYLA restarts train two layers. The classical reference is used for evaluation, not as a replacement sampled answer.

Sources: [IBM QAOA](https://quantum.cloud.ibm.com/docs/en/tutorials/quantum-approximate-optimization-algorithm), [DiagonalGate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.DiagonalGate), [bit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering).

In [7]:
%%writefile $VALIDATION_PATH
"""Run local Qiskit circuits, then compare an independent browser engine.

No QiskitRuntimeService, real backends, account keys or remote job submissions.
"""
from pathlib import Path
from copy import deepcopy
import json
import subprocess
import importlib.metadata
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit.library import DiagonalGate
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
import booking_model
import fairness_model

HERE = Path(__file__).resolve().parent


def evolution(energies, angles):
    """Vectorized local training; Qiskit is the independent circuit oracle."""
    size = len(energies)
    state = np.full(size, 1 / np.sqrt(size), dtype=complex)
    normalized = np.asarray(energies) / max(1, max(map(abs, energies)))
    for gamma, beta in np.reshape(angles, (-1, 2)):
        state *= np.exp(-1j * gamma * normalized)
        for q in range(size.bit_length() - 1):
            view = state.reshape((-1, 2, 1 << q))
            a, b = view[:, 0, :].copy(), view[:, 1, :].copy()
            view[:, 0, :] = np.cos(beta) * a - 1j * np.sin(beta) * b
            view[:, 1, :] = np.cos(beta) * b - 1j * np.sin(beta) * a
    return np.abs(state) ** 2


def circuit(model, angles):
    qc = QuantumCircuit(model['num_qubits'])
    qc.h(range(qc.num_qubits))
    scale = max(1, max(map(abs, model['energies'])))
    for gamma, beta in np.reshape(angles, (-1, 2)):
        if model['name'] == 'booking':
            meta = model['metadata']
            for q, coefficient in enumerate(meta['z']):
                qc.rz(2 * gamma * coefficient / scale, q)
            for i, j, coefficient in meta['zz']:
                qc.rzz(2 * gamma * coefficient / scale, i, j)
        else:
            phases = np.exp(-1j * gamma * np.asarray(model['energies']) / scale)
            # Decompose structured diagonal synthesis; never form a dense unitary.
            qc.compose(DiagonalGate(phases).definition.decompose(), inplace=True)
        qc.rx(2 * beta, range(qc.num_qubits))
    return qc


def train(model):
    energies = np.asarray(model['energies'])
    scale = max(1, float(max(abs(energies))))
    def loss(angles):
        return float(evolution(energies, angles) @ energies / scale)
    rng = np.random.default_rng(7)
    trials = [minimize(loss, rng.uniform(0, 2*np.pi, 4), method='COBYLA',
                       options={'maxiter':120, 'rhobeg':1}) for _ in range(3)]
    result = min(trials, key=lambda r:r.fun)
    return result.x.tolist(), {'method':'COBYLA', 'restarts':3,
        'max_evaluations_per_restart':120, 'training_seed':7,
        'evaluations':sum(int(r.nfev) for r in trials)}


def main():
    math_audit={'booking':booking_model.verify(), 'fairness':fairness_model.verify()}
    out = HERE / 'web'
    out.mkdir(exist_ok=True)
    definitions = [('booking', booking_model, None),
                   ('booking-less-waiting', booking_model, {'waiting_weight':1}),
                   ('fairness', fairness_model, None)]
    slow = deepcopy(fairness_model.DEFAULT)
    slow['cars'][1]['fast'] = False
    definitions.append(('fairness-slow-b', fairness_model, slow))
    cases, summaries, models = [], [], []
    for identifier, module, scenario in definitions:
        model = module.build_model(scenario)
        angles, training = train(model)
        qc = circuit(model, angles)
        p = Statevector.from_instruction(qc).probabilities()
        custom = evolution(model['energies'], angles)
        error = float(np.max(np.abs(p-custom)))
        assert error < 1e-10, (identifier,error)
        measured = qc.copy(); measured.measure_all()
        result = StatevectorSampler(seed=7).run([measured], shots=2048).result()[0]
        counts = result.data.meas.get_counts()
        assert sum(counts.values()) == 2048
        valid_shots = sum(v for bits,v in counts.items() if model['feasible'][int(bits,2)])
        valid = [int(bits,2) for bits in counts if model['feasible'][int(bits,2)]]
        best = min(valid,key=lambda i:model['energies'][i]) if valid else None
        optimum = min(e for i,e in enumerate(model['energies']) if model['feasible'][i])
        summary = {'id':identifier, 'qubits':model['num_qubits'], 'shots':2048,
            'execution':'local Qiskit StatevectorSampler; no hardware',
            'valid_shots':valid_shots,'best_sample_index':best,
            'sampled_optimum':best is not None and abs(model['energies'][best]-optimum)<1e-8,
            'best_sample':module.decode(best,scenario) if best is not None else None,
            'feasible_probability':float(sum(v for i,v in enumerate(p) if model['feasible'][i])),
            'optimum_probability':float(sum(v for i,v in enumerate(p) if model['feasible'][i] and abs(model['energies'][i]-optimum)<1e-8)),
            'uniform_feasible_probability':sum(model['feasible'])/len(p),
            'qiskit_numpy_max_error':error, 'training':training,
            'gate_counts':dict(qc.count_ops())}
        summaries.append(summary)
        for settings in (angles,[0.37,0.51,0.79,0.23],[0,0,0,0]):
            reference = Statevector.from_instruction(circuit(model,settings)).probabilities()
            cases.append({'id':identifier,'model':model,'angles':settings,'probabilities':reference.tolist()})
        model.update(id=identifier,angles=angles,training=training,
            decoded={str(i):module.decode(i,scenario) for i,allowed in enumerate(model['feasible']) if allowed})
        models.append(model)
        print(json.dumps({k:summary[k] for k in ['id','qubits','valid_shots','sampled_optimum','qiskit_numpy_max_error']}),flush=True)
    (HERE/'comparison.json').write_text(json.dumps(cases))
    browser = subprocess.run(['node',str(HERE/'compare.mjs'),str(HERE/'comparison.json')],check=True,capture_output=True,text=True)
    print(browser.stdout,end='')
    browser_metrics=json.loads(browser.stdout.splitlines()[0])
    versions = {name:importlib.metadata.version(name) for name in ['qiskit','numpy','scipy','qiskit-aer','qiskit-ibm-runtime']}
    report={'schema_version':1,'scope':'bounded local circuit simulation; no quantum advantage claim',
            'versions':versions,'mathematical_validation':math_audit,'results':summaries, 'browser_comparison':browser_metrics,
            'browser_max_error':browser_metrics['max_probability_error']}
    (out/'models.json').write_text(json.dumps(models))
    (out/'validation.json').write_text(json.dumps(report,indent=2))
    print(json.dumps(versions));print('SIMULATIONS_VERIFIED')


if __name__ == '__main__':
    main()


Writing temporary-source/validate.py


In [8]:
from validate import circuit, evolution, train
from copy import deepcopy
import numpy as np
import qiskit, scipy
assert qiskit.__version__==REQUIRED['qiskit'] and np.__version__==REQUIRED['numpy'] and scipy.__version__==REQUIRED['scipy'], 'Restart the kernel to load the installed package versions.'
from qiskit.quantum_info import Statevector
from qiskit.primitives import StatevectorSampler
slow=deepcopy(fairness_model.DEFAULT)
slow['cars'][1]['fast']=False
EXPERIMENTS=[('booking',booking_model,None),
             ('booking-less-waiting',booking_model,{'waiting_weight':1}),
             ('fairness',fairness_model,None),
             ('fairness-slow-b',fairness_model,slow)]
RUNS=[]; COMPARISONS=[]
for identifier,module,scenario in EXPERIMENTS:
    model=module.build_model(scenario)
    angles,training=train(model)
    # Check optimized, arbitrary and zero angles, not only the selected winner.
    for settings in (angles,[0.37,0.51,0.79,0.23],[0,0,0,0]):
        p=Statevector.from_instruction(circuit(model,settings)).probabilities()
        error=float(np.max(np.abs(p-evolution(model['energies'],settings))))
        assert error<1e-10, (identifier,error)
        COMPARISONS.append({'id':identifier,'model':model,'angles':settings,'probabilities':p.tolist()})
    qc=circuit(model,angles)
    p=Statevector.from_instruction(qc).probabilities()
    qc.measure_all()
    counts=StatevectorSampler(seed=7).run([qc],shots=2048).result()[0].data.meas.get_counts()
    assert sum(counts.values())==2048
    valid=[int(bits,2) for bits in counts if model['feasible'][int(bits,2)]]
    best=min(valid,key=lambda i:model['energies'][i]) if valid else None
    exact=min(e for i,e in enumerate(model['energies']) if model['feasible'][i])
    run={'id':identifier,'execution':'local Qiskit ideal statevector simulation',
         'scenario':model['metadata'].get('scenario',{'waiting_weight':model['metadata'].get('waiting_weight'),'booking_options':model['metadata'].get('options')}),'angles':angles,'training':training,'shots':2048,'seed':7,
         'valid_shots':sum(v for bits,v in counts.items() if model['feasible'][int(bits,2)]),
         'counts':counts,'best_sample_index':best,
         'best_sample':module.decode(best,scenario) if best is not None else None,
         'sampled_optimum':best is not None and abs(model['energies'][best]-exact)<1e-8,
         'feasible_probability':sum(float(v) for i,v in enumerate(p) if model['feasible'][i]),
         'optimum_probability':sum(float(v) for i,v in enumerate(p) if model['feasible'][i] and abs(model['energies'][i]-exact)<1e-8),
         'uniform_feasibility':sum(model['feasible'])/len(p)}
    RUNS.append(run)
    print(identifier, 'valid shots:',run['valid_shots'],'/ 2048; optimum sampled:',run['sampled_optimum'])
print('QISKIT_CIRCUITS_VERIFIED')

booking valid shots: 428 / 2048; optimum sampled: True
booking-less-waiting valid shots: 461 / 2048; optimum sampled: True


fairness valid shots: 1023 / 2048; optimum sampled: True


fairness-slow-b valid shots: 718 / 2048; optimum sampled: True
QISKIT_CIRCUITS_VERIFIED


## 7. Inspect the answers honestly

The ideal sampler and browser sampler use different random-number generators, so their seeded counts need not be identical. Compare probabilities and decoded objectives. An X mixer can generate invalid states, and shallow QAOA can miss the optimum. A failed run remains a failed run.

In [9]:
rows=['| Experiment | Valid shots | P(valid) | P(optimum) | Uniform P(valid) | Sampled optimum? |',
      '|---|---:|---:|---:|---:|---|']
for r in RUNS:
    rows.append(f"| {r['id']} | {r['valid_shots']}/2048 | {r['feasible_probability']:.2%} | {r['optimum_probability']:.2%} | {r['uniform_feasibility']:.2%} | {r['sampled_optimum']} |")
display(Markdown('\n'.join(rows)))
for r in RUNS:
    display(Markdown('### '+r['id']))
    if r['best_sample'] is None:
        print('No feasible schedule sampled. No classical answer substituted.')
    else:
        print(json.dumps(r['best_sample'],indent=2))

| Experiment | Valid shots | P(valid) | P(optimum) | Uniform P(valid) | Sampled optimum? |
|---|---:|---:|---:|---:|---|
| booking | 428/2048 | 21.55% | 7.43% | 9.38% | True |
| booking-less-waiting | 461/2048 | 22.47% | 8.32% | 9.38% | True |
| fairness | 1023/2048 | 50.53% | 0.89% | 22.78% | True |
| fairness-slow-b | 718/2048 | 34.91% | 1.49% | 8.98% | True |

### booking

{
  "valid": true,
  "feasible": true,
  "reasons": [],
  "bits": [
    0,
    1,
    0,
    0,
    1
  ],
  "booking_bitstring": "01001",
  "qiskit_bitstring": "10010",
  "schedule": [
    {
      "vehicle_id": "A",
      "station_id": "North",
      "slot_id": "late",
      "start_minute": 540,
      "end_minute": 600,
      "arrival_minute": 475,
      "arrival_energy_kwh": 14.6,
      "charging_energy_kwh": 10.4,
      "completion_minute": 596.7272727272727,
      "electricity_cost_rupees": 10.4,
      "waiting_minutes": 65,
      "reasons": []
    },
    {
      "vehicle_id": "B",
      "station_id": "North",
      "slot_id": "early",
      "start_minute": 480,
      "end_minute": 540,
      "arrival_minute": 475,
      "arrival_energy_kwh": 5.8,
      "charging_energy_kwh": 9.2,
      "completion_minute": 530.1818181818181,
      "electricity_cost_rupees": 18.4,
      "waiting_minutes": 5,
      "reasons": []
    }
  ],
  "score": 46.3,
  "penalty": 0.0,
  "energy": 46.3,
  "tota

### booking-less-waiting

{
  "valid": true,
  "feasible": true,
  "reasons": [],
  "bits": [
    0,
    0,
    1,
    0,
    1
  ],
  "booking_bitstring": "00101",
  "qiskit_bitstring": "10100",
  "schedule": [
    {
      "vehicle_id": "A",
      "station_id": "South",
      "slot_id": "early",
      "start_minute": 480,
      "end_minute": 540,
      "arrival_minute": 480,
      "arrival_energy_kwh": 14.0,
      "charging_energy_kwh": 11.0,
      "completion_minute": 540.0,
      "electricity_cost_rupees": 33.0,
      "waiting_minutes": 0,
      "reasons": []
    },
    {
      "vehicle_id": "B",
      "station_id": "North",
      "slot_id": "early",
      "start_minute": 480,
      "end_minute": 540,
      "arrival_minute": 475,
      "arrival_energy_kwh": 5.8,
      "charging_energy_kwh": 9.2,
      "completion_minute": 530.1818181818181,
      "electricity_cost_rupees": 18.4,
      "waiting_minutes": 5,
      "reasons": []
    }
  ],
  "score": 56.4,
  "penalty": 0.0,
  "energy": 56.4,
  "total_electricit

### fairness

{
  "index": 2922,
  "bits": "101101101010",
  "slot_codes": [
    2,
    5,
    5,
    5
  ],
  "feasible": true,
  "schedule": [
    [
      50,
      0
    ],
    [
      0,
      50
    ],
    [
      0,
      50
    ],
    [
      0,
      50
    ]
  ],
  "reason": null,
  "energy": [
    12.5,
    37.5
  ],
  "final": [
    80.83333333333333,
    76.875
  ],
  "minimum": 76.875,
  "total": 50.0,
  "cost": 400.0,
  "total_units": 8,
  "objective_exact": [
    "615/8",
    "50"
  ]
}


### fairness-slow-b

{
  "index": 2276,
  "bits": "100011100100",
  "slot_codes": [
    4,
    4,
    3,
    4
  ],
  "feasible": true,
  "schedule": [
    [
      25,
      25
    ],
    [
      25,
      25
    ],
    [
      0,
      25
    ],
    [
      25,
      25
    ]
  ],
  "reason": null,
  "energy": [
    18.75,
    25.0
  ],
  "final": [
    91.25,
    61.25
  ],
  "minimum": 61.25,
  "total": 43.75,
  "cost": 350.0,
  "total_units": 7,
  "objective_exact": [
    "245/4",
    "175/4"
  ]
}


## 8. Optional browser-engine parity check

The complete JavaScript engine is included here. If Node.js is available, execute the same 12 settings and assert probability agreement with Qiskit. If Node is unavailable (as on some online notebook hosts), this cell explicitly reports that check as skipped; the Python/Qiskit checks above still run.

In [10]:
%%writefile $ENGINE_PATH
// Little-endian statevector QAOA. This is circuit simulation, not a QPU client.
export function probabilities(energies, angles) {
  const size=energies.length, n=Math.log2(size);
  if(!Number.isInteger(n)||n<1||n>12||!energies.every(Number.isFinite))throw Error('Expected a finite diagonal with 2–4096 entries.');
  if(!Array.isArray(angles)||angles.length<2||angles.length>6||angles.length%2||!angles.every(Number.isFinite))throw Error('Use one to three [gamma, beta] pairs.');
  const scale=Math.max(1,...energies.map(Math.abs));
  const re=new Float64Array(size).fill(1/Math.sqrt(size)), im=new Float64Array(size);
  for(let layer=0;layer<angles.length;layer+=2){
    const gamma=angles[layer],beta=angles[layer+1];
    for(let i=0;i<size;i++){
      const phase=-gamma*energies[i]/scale,c=Math.cos(phase),s=Math.sin(phase),r=re[i],v=im[i];
      re[i]=c*r-s*v; im[i]=s*r+c*v;
    }
    const c=Math.cos(beta),s=Math.sin(beta);
    for(let q=0;q<n;q++)for(let start=0;start<size;start+=2**(q+1))for(let j=0;j<2**q;j++){
      const a=start+j,b=a+2**q,ar=re[a],ai=im[a],br=re[b],bi=im[b];
      re[a]=c*ar+s*bi; im[a]=c*ai-s*br;
      re[b]=c*br+s*ai; im[b]=c*bi-s*ar;
    }
  }
  return Array.from(re,(v,i)=>v*v+im[i]*im[i]);
}

export function sample(probs,shots=2048,seed=7){
  if(!Number.isInteger(shots)||shots<1||shots>100000||!Number.isInteger(seed)||seed<0||seed>0xffffffff)throw Error('Invalid shots or seed.');
  if(!Array.isArray(probs)||probs.some(p=>!Number.isFinite(p)||p<0)||Math.abs(probs.reduce((a,b)=>a+b,0)-1)>1e-8)throw Error('Invalid probability distribution.');
  const cumulative=[];let total=0,state=seed>>>0;
  for(const p of probs)cumulative.push(total+=p);
  const counts={};
  for(let shot=0;shot<shots;shot++){
    state=(state+0x6D2B79F5)>>>0;let t=state;
    t=Math.imul(t^(t>>>15),t|1);t^=t+Math.imul(t^(t>>>7),t|61);
    const draw=((t^(t>>>14))>>>0)/4294967296;let lo=0,hi=probs.length-1;
    while(lo<hi){const mid=(lo+hi)>>>1;if(cumulative[mid]>draw)hi=mid;else lo=mid+1;}
    counts[lo]=(counts[lo]||0)+1;
  }
  return counts;
}

export function summarize(model,probs,counts){
  const entries=Object.entries(counts).map(([i,count])=>({index:Number(i),count}));
  const valid=entries.filter(e=>model.feasible[e.index]);
  const best=valid.reduce((a,b)=>!a||model.energies[b.index]<model.energies[a.index]?b:a,null);
  const optimum=Math.min(...model.energies.filter((_,i)=>model.feasible[i]));
  const shots=entries.reduce((s,e)=>s+e.count,0);
  return {shots,valid_shots:valid.reduce((s,e)=>s+e.count,0),best_sample_index:best?.index??null,
    sampled_optimum:best!==null&&Math.abs(model.energies[best.index]-optimum)<1e-8,
    feasible_probability:probs.reduce((s,p,i)=>s+(model.feasible[i]?p:0),0),
    optimum_probability:probs.reduce((s,p,i)=>s+(model.feasible[i]&&Math.abs(model.energies[i]-optimum)<1e-8?p:0),0),
    uniform_feasible_probability:model.feasible.filter(Boolean).length/probs.length};
}


Writing temporary-source/engine.mjs


In [11]:
%%writefile $COMPARE_PATH
import {readFileSync} from 'node:fs';
import assert from 'node:assert/strict';
import {probabilities,sample,summarize} from './engine.mjs';
const cases=JSON.parse(readFileSync(process.argv[2],'utf8'));let maxError=0;
for(const test of cases){
  const p=probabilities(test.model.energies,test.angles);
  const err=Math.max(...p.map((v,i)=>Math.abs(v-test.probabilities[i])));
  assert.ok(err<1e-10,`${test.id}: probability error ${err}`);maxError=Math.max(maxError,err);
  assert.ok(Math.abs(p.reduce((a,b)=>a+b,0)-1)<1e-10);
  const counts=sample(p,2048,7);assert.deepEqual(counts,sample(p,2048,7));
  assert.equal(Object.values(counts).reduce((a,b)=>a+b,0),2048);
  assert.equal(summarize(test.model,p,counts).shots,2048);
  // Negative controls: uniform cost produces a uniform distribution even after mixing.
  const uniform=probabilities(p.map(()=>0),test.angles);assert.ok(uniform.every(v=>Math.abs(v-1/p.length)<1e-10));
}
for(const bad of [[1,2,3],[1,NaN]])assert.throws(()=>probabilities(bad,[1,1]));
assert.throws(()=>sample([0.2,0.2]));
// Invalid samples must not silently turn into the exact answer.
assert.equal(summarize({energies:[0,1],feasible:[true,false]},[0,1],{1:10}).best_sample_index,null);
console.log(JSON.stringify({cases:cases.length,max_probability_error:maxError}));
console.log('BROWSER_VERIFIED');


Writing temporary-source/compare.mjs


In [12]:
node=shutil.which('node')
BROWSER_CHECK=None
if node:
    comparison_file=SIM_DIR/'comparison.json'
    comparison_file.write_text(json.dumps(COMPARISONS))
    completed=subprocess.run([node,COMPARE_PATH,str(comparison_file)],check=True,capture_output=True,text=True)
    print(completed.stdout)
    BROWSER_CHECK=json.loads(completed.stdout.splitlines()[0])
else:
    print('SKIPPED: Node.js is unavailable; browser parity was not rerun in this runtime.')

{"cases":12,"max_probability_error":3.0097452308197603e-16}
BROWSER_VERIFIED



## 9. Optional fake-device noise simulation

This booking-only smoke test uses **FakeManilaV2**, a bundled five-qubit snapshot, with Aer. It does not instantiate a cloud service or fetch a real backend. Snapshot noise approximates some device errors; it is not current hardware evidence or a fairness-noise benchmark. See [IBM local testing mode](https://quantum.cloud.ibm.com/docs/en/guides/local-testing-mode).

Set `RUN_FAKE_DEVICE=False` to skip this optional local test.

In [13]:
RUN_FAKE_DEVICE=True
FAKE_RESULT=None
if RUN_FAKE_DEVICE:
    from qiskit import transpile
    from qiskit_aer import AerSimulator
    from qiskit_ibm_runtime.fake_provider import FakeManilaV2
    model=booking_model.build_model()
    qc=circuit(model,RUNS[0]['angles']);qc.measure_all()
    fake_sim=AerSimulator.from_backend(FakeManilaV2(),method='density_matrix')
    compiled=transpile(qc,backend=fake_sim,optimization_level=1,seed_transpiler=7)
    noisy_counts=fake_sim.run(compiled,shots=256,seed_simulator=7).result().get_counts()
    assert sum(noisy_counts.values())==256
    FAKE_RESULT={'execution':'local Aer + FakeManilaV2 snapshot', 'shots':256,
        'seed':7,'counts':noisy_counts,
        'valid_shots':sum(v for bits,v in noisy_counts.items() if model['feasible'][int(bits,2)])}
    print(json.dumps(FAKE_RESULT,indent=2))
    print('FAKE_DEVICE_SIMULATION_VERIFIED')
else:
    print('SKIPPED: optional fake-device noise test.')

{
  "execution": "local Aer + FakeManilaV2 snapshot",
  "shots": 256,
  "seed": 7,
  "counts": {
    "11111": 2,
    "01001": 3,
    "01100": 4,
    "11110": 3,
    "01111": 2,
    "00011": 2,
    "10010": 10,
    "11010": 2,
    "00001": 16,
    "10101": 8,
    "00101": 9,
    "10001": 13,
    "01000": 9,
    "01110": 2,
    "10011": 5,
    "10111": 3,
    "01101": 1,
    "10100": 16,
    "00111": 2,
    "11000": 18,
    "00100": 17,
    "11011": 2,
    "10000": 29,
    "11001": 5,
    "11100": 3,
    "01010": 4,
    "11101": 2,
    "00110": 3,
    "10110": 10,
    "00010": 15,
    "00000": 36
  },
  "valid_shots": 44
}
FAKE_DEVICE_SIMULATION_VERIFIED


## 10. Export reproducible evidence

The JSON contains inputs, parameters, counts, versions, audit results and observed outcomes. The notebook's saved cell outputs are a record of one executed run; rerunning recomputes them. Neither matching the optimum nor outperforming unrestricted random feasibility establishes speedup over a strong classical solver.

Assumptions: fixed dock presence, discrete charging blocks, constant power and ideal efficiency. Real charging curves, losses, thermal behavior, queues, uncertain arrivals and physical charger control remain outside this model.

In [14]:
EVIDENCE={'schema_version':1,'execution':'local simulation only; no real QPU',
          'versions':{name:installed(name) for name in REQUIRED},
          'mathematical_checks':{'booking':booking_checks,'fairness':fairness_checks},
          'experiments':RUNS,'browser_check':BROWSER_CHECK,'fake_device_check':FAKE_RESULT}
OUTPUT=Path.cwd()/'ev-scheduling-validation-results.json'
OUTPUT.write_text(json.dumps(EVIDENCE,indent=2))
print('Saved ev-scheduling-validation-results.json')
assert len(RUNS)==4 and len(COMPARISONS)==12
print('EV_NOTEBOOK_VERIFIED')
sys.path.remove(str(SIM_DIR))
WORKSPACE.cleanup()

Saved ev-scheduling-validation-results.json
EV_NOTEBOOK_VERIFIED
